# Question 2 — Signal Generation
Algorithmic Trading Winter Project — Finance And Analytics Club, IIT Kanpur

This notebook takes each technical indicator built in Question 1 and defines a corresponding **signal-generating function**. Every signal function returns the dataset with an added `Signal` column: `1` = Buy, `-1` = Sell, `0` = No signal. At the end, the number of buy/sell signals produced by each indicator is printed.

## Setup: download data and re-build indicator functions (from Q1)

In [6]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ticker = "TCS.NS"
data = yf.download(ticker, start='2023-01-01', end='2023-12-31')

# ---------------- Indicator functions (from Question 1) ----------------

def sma(dat, period: int):
    length = len(dat.index)
    smas = []
    for i in range(0, period - 1):
        smas.append(None)
    for i in range(period - 1, length):
        sumbox = []
        for j in range(0, period):
            a = float(dat.iloc[i - j]['Close'][ticker])
            sumbox.append(a)
        smas.append(sum(sumbox) / period)
    dat[str(period) + '-day SMA'] = smas
    return dat

def std_dev(dat, column: str, period: int):
    length = len(dat.index)
    std_devs = []
    for i in range(0, period - 1):
        std_devs.append(None)
    for i in range(period - 1, length):
        dataset1 = []
        for j in range(0, period):
            a = dat.iloc[i - j][column]
            if isinstance(a, pd.Series):
                a = a.iloc[0]
            dataset1.append(a)
        avg = sum(dataset1) / len(dataset1)
        sumbo = [(x - avg) ** 2 for x in dataset1]
        std_devs.append((sum(sumbo) / len(sumbo)) ** 0.5)
    dat['Std Dev of ' + column] = std_devs
    return dat

def stochast(dat, period: int):
    length = len(dat.index)
    stochasts = []
    for i in range(0, period - 1):
        stochasts.append(None)
    for i in range(period - 1, length):
        valueshigh, valueslow = [], []
        for j in range(0, period):
            valueshigh.append(float(dat.iloc[i - j]['High'][ticker]))
            valueslow.append(float(dat.iloc[i - j]['Low'][ticker]))
        highvalue, lowvalue = max(valueshigh), min(valueslow)
        currentvalue = float(dat.iloc[i]['Close'][ticker])
        stochasts.append(((currentvalue - lowvalue) / (highvalue - lowvalue)) * 100)
    dat[str(period) + '-day SO'] = stochasts
    return dat

def rsi(dat, period: int):
    length = len(dat.index)
    gainloss = [0]
    for i in range(1, length):
        a = float(dat.iloc[i - 1]['Close'][ticker])
        b = float(dat.iloc[i]['Close'][ticker])
        gainloss.append(b - a)
    gains, losses = [], []
    for i in gainloss:
        if i >= 0:
            gains.append(i); losses.append(0)
        else:
            gains.append(0); losses.append(abs(i))
    rsis = []
    for i in range(0, period - 1):
        rsis.append(None)
    for i in range(period - 1, length):
        vg = [gains[i - j] for j in range(period)]
        vl = [losses[i - j] for j in range(period)]
        avggain, avgloss = sum(vg) / period, sum(vl) / period
        if avgloss == 0:
            rsis.append(100)
            continue
        rs = avggain / avgloss
        rsis.append(100 - (100 / (1 + rs)))
    dat[str(period) + '-day RSI'] = rsis
    return dat

def ema(dat, period: int):
    length = len(dat.index)
    emas = []
    for i in range(0, period - 1):
        emas.append(None)
    sumbox = [float(dat.iloc[i]['Close'][ticker]) for i in range(period)]
    emas.append(sum(sumbox) / len(sumbox))
    multiplier = 2 / (period + 1)
    for i in range(period, length):
        a = float(dat.iloc[i]['Close'][ticker])
        b = emas[i - 1]
        emas.append(a * multiplier + b * (1 - multiplier))
    dat[str(period) + '-day EMA'] = emas
    return dat

def macd(dat1, fastperiod: int, slowperiod: int, signalsmoothper: int):
    dat1 = ema(dat1, fastperiod)
    dat1 = ema(dat1, slowperiod)
    macd_vals = np.array(dat1[str(fastperiod) + '-day EMA'], dtype=float) - \
                np.array(dat1[str(slowperiod) + '-day EMA'], dtype=float)
    dat1['MACD ({0},{1})'.format(fastperiod, slowperiod)] = macd_vals
    dat1[str(signalsmoothper) + '-day SMA of MACD'] = dat1['MACD ({0},{1})'.format(fastperiod, slowperiod)].rolling(signalsmoothper).mean()
    return dat1

def atr(dat, period: int):
    length = len(dat.index)
    h_l, h_cp, l_cp = [], [None], [None]
    x, y = float(dat.iloc[0]['High'][ticker]), float(dat.iloc[0]['Low'][ticker])
    h_l.append(x - y)
    for i in range(1, length):
        a = float(dat.iloc[i]['High'][ticker])
        b = float(dat.iloc[i]['Low'][ticker])
        c = float(dat.iloc[i - 1]['Close'][ticker])
        h_l.append(a - b); h_cp.append(abs(a - c)); l_cp.append(abs(b - c))
    trs = [h_l[0]]
    for i in range(1, length):
        trs.append(max(h_l[i], h_cp[i], l_cp[i]))
    atrs = []
    for i in range(0, period - 1):
        sumbox = trs[0:i]
        atrs.append(sum(sumbox) / (i + 1) if i > 0 else trs[0])
    for i in range(period - 1, length):
        m, n = atrs[i - 1], trs[i]
        atrs.append(((period - 1) * m + n) / period)
    dat[str(period) + '-day ATR'] = atrs
    return dat

def adm(dat1, period: int):
    length = len(dat1.index)
    dat1 = atr(dat1, period)
    h_ph, l_pl = [None], [None]
    for i in range(1, length):
        a = float(dat1.iloc[i]['High'][ticker])
        b = float(dat1.iloc[i - 1]['High'][ticker])
        c = float(dat1.iloc[i]['Low'][ticker])
        d = float(dat1.iloc[i - 1]['Low'][ticker])
        h_ph.append(a - b); l_pl.append(d - c)
    plusdm, minusdm = [None], [None]
    for i in range(1, length):
        if h_ph[i] > l_pl[i] and h_ph[i] > 0:
            plusdm.append(h_ph[i]); minusdm.append(0)
        elif l_pl[i] > h_ph[i] and l_pl[i] > 0:
            plusdm.append(0); minusdm.append(l_pl[i])
        else:
            plusdm.append(0); minusdm.append(0)
    dat1['+DM'], dat1['-DM'] = plusdm, minusdm
    dat1['Smooth+DM'] = dat1['+DM'].rolling(period).mean()
    dat1['Smooth-DM'] = dat1['-DM'].rolling(period).mean()
    dat1['+DI'] = (dat1['Smooth+DM'] / dat1[str(period) + '-day ATR']) * 100
    dat1['-DI'] = (dat1['Smooth-DM'] / dat1[str(period) + '-day ATR']) * 100
    dat1['DX'] = abs((dat1['+DI'] - dat1['-DI']) / (dat1['+DI'] + dat1['-DI'])) * 100
    dat1[str(period) + '-day ADX'] = dat1['DX'].rolling(period).mean()
    dat1 = dat1.drop(columns=['DX', '+DI', '-DI', 'Smooth+DM', 'Smooth-DM', '+DM', '-DM'])
    return dat1

# Compute every indicator once so all signal functions below have what they need
data = sma(data, 20)
data = std_dev(data, 'Close', 20)
data = stochast(data, 14)
data = rsi(data, 14)
data = macd(data, 12, 26, 9)
data = adm(data, 14)

data.tail()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume,20-day SMA,Std Dev of Close,14-day SO,14-day RSI,12-day EMA,26-day EMA,"MACD (12,26)",9-day SMA of MACD,14-day ATR,14-day ADX
Ticker,TCS.NS,TCS.NS,TCS.NS,TCS.NS,TCS.NS,,,,,,,,,,
Date,,,,,,,,,,,,,,,
2023-12-22,3513.734131,3533.903140,3456.764592,3491.681406,2413058,3346.213916,122.436496,75.363631,74.704574,3430.511253,3346.402916,84.108337,68.170561,68.574686,51.228367
2023-12-26,3487.592285,3522.922541,3482.630288,3509.920712,1285231,3361.763367,119.624052,66.955448,70.625886,3439.292951,3356.861388,82.431563,72.054720,66.554512,51.864256
2023-12-27,3501.972900,3508.404946,3462.278038,3490.762809,1293976,3377.432288,116.395710,67.814228,67.761543,3448.936020,3367.610389,81.325631,76.049360,65.095398,51.553652
2023-12-28,3491.589355,3526.598145,3484.422400,3513.734056,1682889,3390.578931,113.666209,64.726686,65.852543,3455.498071,3376.794016,78.704055,79.392758,63.458280,52.460873
2023-12-29,3485.616699,3512.447693,3459.888521,3484.330380,1574996,3404.628308,106.983012,62.950719,64.415360,3460.131706,3384.854955,75.276751,80.556904,62.679772,52.592435


## Bollinger Bands Signal
Mean-reversion logic: price touching/crossing the **lower band** is a Buy (oversold); price touching/crossing the **upper band** is a Sell (overbought).

In [7]:
def bollinger_signal(dat, period: int = 20):
    upper = dat[str(period) + '-day SMA'] + 2 * dat['Std Dev of Close']
    lower = dat[str(period) + '-day SMA'] - 2 * dat['Std Dev of Close']
    price = dat['Close'][ticker]
    signal = []
    for p, u, l in zip(price, upper, lower):
        if pd.isna(u) or pd.isna(l):
            signal.append(0)
        elif p <= l:
            signal.append(1)    # Buy: price at/below lower band
        elif p >= u:
            signal.append(-1)   # Sell: price at/above upper band
        else:
            signal.append(0)
    dat['Bollinger Signal'] = signal
    return dat

data = bollinger_signal(data, 20)
buy = (data['Bollinger Signal'] == 1).sum()
sell = (data['Bollinger Signal'] == -1).sum()
print(f"Bollinger Bands -> Buy signals: {buy}, Sell signals: {sell}")

Bollinger Bands -> Buy signals: 8, Sell signals: 15


## MACD Signal
Classic crossover logic: MACD line crossing **above** the signal line is a Buy; crossing **below** is a Sell.

In [8]:
def macd_signal(dat, fastperiod: int = 12, slowperiod: int = 26, signalsmoothper: int = 9):
    macd_line = dat['MACD ({0},{1})'.format(fastperiod, slowperiod)]
    signal_line = dat[str(signalsmoothper) + '-day SMA of MACD']
    signal = [0]
    for i in range(1, len(dat)):
        if pd.isna(signal_line.iloc[i]) or pd.isna(signal_line.iloc[i - 1]):
            signal.append(0)
        elif macd_line.iloc[i - 1] <= signal_line.iloc[i - 1] and macd_line.iloc[i] > signal_line.iloc[i]:
            signal.append(1)    # bullish crossover
        elif macd_line.iloc[i - 1] >= signal_line.iloc[i - 1] and macd_line.iloc[i] < signal_line.iloc[i]:
            signal.append(-1)   # bearish crossover
        else:
            signal.append(0)
    dat['MACD Signal'] = signal
    return dat

data = macd_signal(data, 12, 26, 9)
buy = (data['MACD Signal'] == 1).sum()
sell = (data['MACD Signal'] == -1).sum()
print(f"MACD -> Buy signals: {buy}, Sell signals: {sell}")

MACD -> Buy signals: 9, Sell signals: 9


## Stochastic Oscillator Signal
Standard thresholds: SO **<= 20** is oversold (Buy), SO **>= 80** is overbought (Sell).

In [9]:
def stochastic_signal(dat, period: int = 14, oversold: float = 20, overbought: float = 80):
    so = dat[str(period) + '-day SO']
    signal = []
    for v in so:
        if pd.isna(v):
            signal.append(0)
        elif v <= oversold:
            signal.append(1)
        elif v >= overbought:
            signal.append(-1)
        else:
            signal.append(0)
    dat['SO Signal'] = signal
    return dat

data = stochastic_signal(data, 14)
buy = (data['SO Signal'] == 1).sum()
sell = (data['SO Signal'] == -1).sum()
print(f"Stochastic Oscillator -> Buy signals: {buy}, Sell signals: {sell}")

Stochastic Oscillator -> Buy signals: 48, Sell signals: 72


## RSI Signal
Standard thresholds: RSI **<= 30** is oversold (Buy), RSI **>= 70** is overbought (Sell).

In [10]:
def rsi_signal(dat, period: int = 14, oversold: float = 30, overbought: float = 70):
    r = dat[str(period) + '-day RSI']
    signal = []
    for v in r:
        if pd.isna(v):
            signal.append(0)
        elif v <= oversold:
            signal.append(1)
        elif v >= overbought:
            signal.append(-1)
        else:
            signal.append(0)
    dat['RSI Signal'] = signal
    return dat

data = rsi_signal(data, 14)
buy = (data['RSI Signal'] == 1).sum()
sell = (data['RSI Signal'] == -1).sum()
print(f"RSI -> Buy signals: {buy}, Sell signals: {sell}")

RSI -> Buy signals: 22, Sell signals: 40


## ADX Signal
ADX alone only measures **trend strength**, not direction, so it can't produce a Buy/Sell signal by itself. The rule used here: when ADX is above a threshold (strong trend, default 25) **and** price rose vs. the previous day -> Buy; strong trend **and** price fell -> Sell. Below the threshold the trend is considered too weak/choppy to trade.

In [11]:
def adx_signal(dat, period: int = 14, threshold: float = 25):
    adx_col = dat[str(period) + '-day ADX']
    price = dat['Close'][ticker]
    signal = [0]
    for i in range(1, len(dat)):
        a = adx_col.iloc[i]
        if pd.isna(a):
            signal.append(0)
            continue
        if a > threshold and price.iloc[i] > price.iloc[i - 1]:
            signal.append(1)    # strong trend, rising price
        elif a > threshold and price.iloc[i] < price.iloc[i - 1]:
            signal.append(-1)   # strong trend, falling price
        else:
            signal.append(0)
    dat['ADX Signal'] = signal
    return dat

data = adx_signal(data, 14)
buy = (data['ADX Signal'] == 1).sum()
sell = (data['ADX Signal'] == -1).sum()
print(f"ADX -> Buy signals: {buy}, Sell signals: {sell}")

ADX -> Buy signals: 86, Sell signals: 84


## ATR Signal
ATR is also a pure volatility measure with no direction of its own, so a **breakout** rule is used: if today's price move exceeds yesterday's ATR (a move bigger than "typical" daily volatility) to the upside -> Buy; to the downside -> Sell.

In [12]:
def atr_signal(dat, period: int = 14, multiplier: float = 1.0):
    atr_col = dat[str(period) + '-day ATR']
    price = dat['Close'][ticker]
    signal = [0]
    for i in range(1, len(dat)):
        a = atr_col.iloc[i - 1]
        if pd.isna(a):
            signal.append(0)
            continue
        change = price.iloc[i] - price.iloc[i - 1]
        if change > multiplier * a:
            signal.append(1)    # breakout up
        elif change < -multiplier * a:
            signal.append(-1)   # breakout down
        else:
            signal.append(0)
    dat['ATR Signal'] = signal
    return dat

data = atr_signal(data, 14)
buy = (data['ATR Signal'] == 1).sum()
sell = (data['ATR Signal'] == -1).sum()
print(f"ATR -> Buy signals: {buy}, Sell signals: {sell}")

ATR -> Buy signals: 15, Sell signals: 16


## Summary: Signal Counts Across All Indicators

In [13]:
signal_cols = ['Bollinger Signal', 'MACD Signal', 'SO Signal', 'RSI Signal', 'ADX Signal', 'ATR Signal']
summary = pd.DataFrame({
    'Buy Signals': [(data[c] == 1).sum() for c in signal_cols],
    'Sell Signals': [(data[c] == -1).sum() for c in signal_cols],
}, index=signal_cols)
summary

,Buy Signals,Sell Signals
Bollinger Signal,8,15
MACD Signal,9,9
SO Signal,48,72
RSI Signal,22,40
ADX Signal,86,84
ATR Signal,15,16
